This is a notebook to run the landcover validation and most of the prerequisite steps to get to this point.
The steps included are:
- downloading landcover dataset zip
- extracting data from the landcover dataset
- running the landcover validation

The steps excluded are the model training and related sanity checks. Because of this, it is required for you to have the unet checkpoint file downloaded already.

All paths in markdown steps are relative to the root of the project (e.g. this notebook is in `./landcover_verification/`). This may not be the case for the Python code boxes.

step 0: download the unet checkpoint file and place it in `./checkpoints/pipeline_best_unet_best.pt`

In [12]:
import os
import hashlib
# assert the file exists
assert os.path.exists("../checkpoints/pipeline_best_unet_best.pt")
# assert file integrity
assert hashlib.md5(open("../checkpoints/pipeline_best_unet_best.pt", "rb").read()).hexdigest() == "d6d073007380e0240b88ca91e4e07895"
print("File integrity verified")

File integrity verified


step 1: download zip of dataset v1 from https://landcover.ai.linuxpolska.com/, extract to `./landcover_verification/datasets/landcover_dataset`

In [1]:
# Download LandCover.ai v1 zip into datasets/, extract to landcover_dataset/
from __future__ import annotations

import shutil
import tempfile
import urllib.request
import zipfile
from pathlib import Path

LANDCOVER_V1_URL = "https://landcover.ai.linuxpolska.com/download/landcover.ai.v1.zip"
ZIP_FILENAME = "landcover.ai.v1.zip"

_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "landcover_verification" else _cwd
DATASETS_DIR = PROJECT_ROOT / "landcover_verification" / "datasets"
TARGET_DIR = DATASETS_DIR / "landcover_dataset"
ZIP_PATH = DATASETS_DIR / ZIP_FILENAME


def _find_images_masks_root(root: Path) -> Path | None:
    if (root / "images").is_dir() and (root / "masks").is_dir():
        return root
    for child in sorted(root.iterdir()):
        if child.is_dir():
            found = _find_images_masks_root(child)
            if found is not None:
                return found
    return None


DATASETS_DIR.mkdir(parents=True, exist_ok=True)

if (TARGET_DIR / "masks").is_dir() and any((TARGET_DIR / "masks").iterdir()):
    print(f"Skip download: {TARGET_DIR} already has masks.")
else:
    print(f"Downloading {LANDCOVER_V1_URL} …")
    urllib.request.urlretrieve(LANDCOVER_V1_URL, ZIP_PATH)
    print(f"Saved {ZIP_PATH}")

    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(tmp_path)
        content_root = _find_images_masks_root(tmp_path)
        if content_root is None:
            raise RuntimeError(
                "Could not find a directory with both 'images/' and 'masks/' inside the zip."
            )
        if TARGET_DIR.exists():
            shutil.rmtree(TARGET_DIR)
        shutil.move(str(content_root), str(TARGET_DIR))

    print(f"Extracted dataset to {TARGET_DIR}")

Saved /home/ehurd1@cfreg.local/ndvi-guided-rgb-segmentation/landcover_verification/datasets/landcover.ai.v1.zip
Extracted dataset to /home/ehurd1@cfreg.local/ndvi-guided-rgb-segmentation/landcover_verification/datasets/landcover_dataset


step 2: